# Does Unemployment Increase During Recessions?

This notebook tests whether the US unemployment rate is higher, and rises faster, during NBER-dated recessions than during expansions.

**Expectation:** Yes. During economic downturns, firms hire less and lay off more workers, so unemployment should rise.

The analysis covers January 2000 to the present, using monthly data from FRED.

## Setup

The libraries used in this notebook:

- **fredapi** pulls series directly from the FRED API.
- **python-dotenv** loads the FRED API key from a local `.env` file. The file is listed in `.gitignore`, so the key never enters the repo.
- **pandas** cleans and merges the data.
- **matplotlib** draws the charts.
- **statsmodels** runs the regressions.

In [4]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.formula.api as smf
from dotenv import load_dotenv
from fredapi import Fred

## Data Pull

Two monthly series come from FRED:

| Series ID | Description | Original source |
|---|---|---|
| `UNRATE` | Civilian unemployment rate (%) | Bureau of Labor Statistics |
| `USREC` | Recession indicator (1 = recession month, 0 = not) | National Bureau of Economic Research |

The pull starts at 1999-12-01, one month before the analysis window. That extra month means the monthly change in unemployment can be computed for January 2000.

In [5]:
# Read the key from .env rather than hard-coding it, so it stays out of the repo.
load_dotenv()
api_key = os.getenv("FRED_API_KEY")
if not api_key:
    raise RuntimeError("FRED_API_KEY not found; add it to the .env file in this folder.")

fred = Fred(api_key=api_key)

START = "1999-12-01"
unrate = fred.get_series("UNRATE", observation_start=START).rename("UNRATE")
usrec = fred.get_series("USREC", observation_start=START).rename("USREC")

In [6]:
display(unrate.head())
display(usrec.head())

1999-12-01    4.0
2000-01-01    4.0
2000-02-01    4.1
2000-03-01    4.0
2000-04-01    3.8
Name: UNRATE, dtype: float64

1999-12-01    0.0
2000-01-01    0.0
2000-02-01    0.0
2000-03-01    0.0
2000-04-01    0.0
Name: USREC, dtype: float64

## Cleaning

The cleaning steps follow the plan:

1. **Merge** UNRATE and USREC into one DataFrame on date.
2. **Compute the monthly change** in the unemployment rate (this month minus last month).
3. **Drop months** where either series is missing.
4. **Convert the recession indicator** to 0/1 integers (FRED returns it as 0.0/1.0).
5. **Restrict** the data to January 2000 onward.

The change is computed on a complete monthly calendar *before* missing months are dropped. FRED has no unemployment value for October 2025 (the government shutdown delayed that month's survey). Computing the change after dropping it would make November 2025's "monthly" change span two months. Computing it first leaves November's change blank, and the averages below skip it. The extra December 1999 month from the pull is used only to compute January 2000's change.

In [7]:
# asfreq("MS") lays the data on a full month-start calendar, so a missing month
# shows up as a gap that diff() won't bridge.
df = pd.concat([unrate, usrec], axis=1).asfreq("MS")
df["UNRATE_CHANGE"] = df["UNRATE"].diff()

df = df.dropna(subset=["UNRATE", "USREC"])
df["USREC"] = df["USREC"].astype(int)
df = df.loc["2000-01-01":]

print(f"{len(df)} months, {df.index.min():%Y-%m} to {df.index.max():%Y-%m}")
df.head()

319 months, 2000-01 to 2026-08


,UNRATE,USREC,UNRATE_CHANGE
2000-01-01,4.0,0,0.0
2000-02-01,4.1,0,0.1
2000-03-01,4.0,0,-0.1
2000-04-01,3.8,0,-0.2
2000-05-01,4.0,0,0.2


In [8]:
summary = (
    df.groupby("USREC")
    .agg(
        months=("UNRATE", "size"),
        avg_unemployment_rate=("UNRATE", "mean"),
        avg_monthly_change=("UNRATE_CHANGE", "mean"),
    )
    .rename(index={0: "Expansion", 1: "Recession"})
    .rename_axis("Period")
)
summary.round(2)

,months,avg_unemployment_rate,avg_monthly_change
Period,,,
Expansion,291,5.53,-0.06
Recession,28,6.43,0.61


Since 2000, the average unemployment rate during recession months (about 6.4%) has been roughly a percentage point higher than during expansion months (about 5.5%). The monthly change shows the bigger difference: during recessions unemployment rose by about 0.6 points per month on average, while during expansions it fell slightly (about −0.06 points per month).

This supports the expectation. The averages also come from only 28 recession months across three recessions (2001, 2007–09 and 2020). The 2020 COVID spike, when unemployment jumped about 10 points in a single month, pulls the recession average up a lot. The regressions later in the notebook test whether these differences are statistically significant.

## Charts

Three charts show the relationship visually:

1. **Unemployment rate over time** with recession months shaded gray. This shows whether unemployment climbs during each recession.
2. **Average unemployment rate** in recessions vs. expansions. This compares the *level*.
3. **Average monthly change in unemployment** in recessions vs. expansions. This compares the *direction*: whether unemployment rises or falls.

A small helper adds the title, axis labels and source note, so every chart is labeled the same way.

In [ ]:
LINE_COLOR = "#2a78d6"
SOURCE_NOTE = "Source: BLS and NBER via FRED."


def label_chart(ax, title, xlabel, ylabel):
    """Apply the title, axis labels and source note every chart in this notebook needs."""
    ax.set_title(title, loc="left", fontsize=13, fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    # Keep the axes and grid quiet so the data stands out.
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e6e5e1", linewidth=0.8)
    ax.set_axisbelow(True)
    # Place the note in axes coordinates so it sits below the x-axis label at any figure size.
    ax.annotate(SOURCE_NOTE, xy=(0, -0.18), xycoords="axes fraction",
                fontsize=9, color="#52514e", va="top")


def label_bars(ax, bars, fmt):
    """Print each bar's value at its tip; negative bars get the label below."""
    for bar in bars:
        height = bar.get_height()
        ax.annotate(fmt.format(height),
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 4 if height >= 0 else -4), textcoords="offset points",
                    ha="center", va="bottom" if height >= 0 else "top", fontsize=10)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

# Shade each recession month as its own span; adjacent months merge into one block.
for month in df.index[df["USREC"] == 1]:
    ax.axvspan(month, month + pd.offsets.MonthBegin(1), color="#d9d8d4", linewidth=0)

ax.plot(df.index, df["UNRATE"], color=LINE_COLOR, linewidth=2)
ax.set_ylim(bottom=0)
ax.set_xlim(df.index.min(), df.index.max())

label_chart(ax, "US unemployment rate since 2000 (recessions shaded gray)",
            "Month", "Unemployment rate (%)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
bars = ax.bar(summary.index, summary["avg_unemployment_rate"], color=LINE_COLOR, width=0.5)
label_bars(ax, bars, "{:.2f}%")
ax.set_ylim(0, summary["avg_unemployment_rate"].max() * 1.15)

label_chart(ax, "Average unemployment rate, 2000–present",
            "Economic period", "Average unemployment rate (%)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(summary.index, summary["avg_monthly_change"], color=LINE_COLOR, width=0.5)
label_bars(ax, bars, "{:+.2f}")
# A zero line makes it clear which bar is a rise and which is a fall.
ax.axhline(0, color="#52514e", linewidth=1)
# Pad both ends by a share of the full range so the value labels fit inside the axes.
low, high = summary["avg_monthly_change"].min(), summary["avg_monthly_change"].max()
pad = 0.15 * (max(high, 0) - min(low, 0))
ax.set_ylim(min(low, 0) - pad, max(high, 0) + pad)

label_chart(ax, "Average monthly change in unemployment, 2000–present",
            "Economic period", "Average monthly change (percentage points)")
plt.tight_layout()
plt.show()

**What the charts show:**

- **Line chart.** Unemployment climbs during or right after each of the three recessions since 2000 (2001, 2007–09 and 2020), then falls slowly through the expansions that follow. The 2020 recession is short but produces by far the largest spike. After the 2001 and 2007–09 recessions, unemployment kept rising for a while after the shaded period ended. Unemployment tends to lag the business cycle, which helps explain why the gap in average levels is smaller than the peaks suggest.
- **Average level.** Unemployment averaged about 6.4% in recession months vs. about 5.5% in expansion months.
- **Average monthly change.** This shows the clearest difference. Unemployment rose about 0.6 points per month during recessions and fell slightly during expansions.

Together, the charts support the expectation that unemployment rises during recessions. The regressions will test whether the differences are statistically significant.